# Notebook 02 - Analisis Fiscal Argentina 2020-2026

**Fuente:** Secretaria de Hacienda | AIF (Sector Publico Base Caja) + IMIG  
**Deflactor:** IPC Nivel General INDEC  
**Unidad:** pesos constantes del ultimo periodo disponible (abril 2026)

**Alcance:**
- Titulares: *Sector Publico Total* = Administracion Nacional + PAMI + Fondos Fiduciarios
- Desglose por componentes: solo Administracion Nacional

**Al finalizar:** se descarga un ZIP con todos los graficos (600 DPI) + Excel de resultados.

In [ ]:
# ── Celda 1: Dependencias ─────────────────────────────────────────────
import sys, io, os, zipfile, requests
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q pandas matplotlib openpyxl

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.dates as mdates
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configuracion global de graficos
DPI       = 600
FIG_W     = 18   # ancho en pulgadas
FIG_H     = 6    # alto en pulgadas
GRAFICOS  = []   # lista de archivos generados

plt.rcParams['figure.dpi']        = 100   # pantalla
plt.rcParams['savefig.dpi']       = DPI
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size']         = 9

def save_fig(fig, nombre):
    """Guarda figura en PNG 600 DPI y registra el archivo."""
    fname = f'{nombre}.png'
    fig.savefig(fname, dpi=DPI, bbox_inches='tight')
    GRAFICOS.append(fname)
    print(f'  Guardado: {fname}')

def fmt_meses(ax):
    """Etiquetas de todos los meses en el eje X, rotadas 45 grados."""
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%Y'))
    ax.tick_params(axis='x', rotation=45)

print('OK')

In [ ]:
# ── Celda 2: Carga de datos ───────────────────────────────────────────
REPO     = 'https://raw.githubusercontent.com/santiagoriverti/cuentas_publicas/main'
REPO_BIN = 'https://github.com/santiagoriverti/cuentas_publicas/raw/main'

df_aif  = pd.read_csv(f'{REPO}/output/aif_consolidado.csv',  parse_dates=['fecha'])
df_imig = pd.read_csv(f'{REPO}/output/imig_consolidado.csv', parse_dates=['fecha'])
df      = df_aif[df_aif['periodo'] == 'mensual'].copy()

ipc_raw = pd.read_excel(
    io.BytesIO(requests.get(f'{REPO_BIN}/data/reference/IPC.xlsx').content),
    usecols=['date', 'Nivel general'])
ipc_raw['date'] = pd.to_datetime(ipc_raw['date'])
ipc = ipc_raw.set_index('date')['Nivel general'].sort_index()
ipc.index = ipc.index.to_period('M').to_timestamp()

BASE     = ipc.index.max()
IPC_BASE = ipc.loc[BASE]

def deflactar(serie):
    idx = serie.index.to_period('M').to_timestamp()
    return serie.values * (IPC_BASE / ipc.reindex(idx).values)

def get_serie(concepto, subsector='total_adm_nacional'):
    mask = (df['concepto_codigo'] == concepto) & (df['subsector'] == subsector)
    return df[mask].set_index('fecha')['valor_millones_pesos'].sort_index()

def get_serie_total(concepto):
    s_nac  = get_serie(concepto, 'total_adm_nacional')
    s_pami = get_serie(concepto, 'pami_fdos_otros')
    s_gen  = get_serie(concepto, 'total_general')
    s_suma = s_nac.add(s_pami.reindex(s_nac.index).fillna(0))
    if len(s_gen) > 0:
        s_suma.update(s_gen)
    return s_suma.sort_index()

print(f'AIF mensual  : {len(df):,} registros | {df.fecha.min().strftime("%Y-%m")} - {df.fecha.max().strftime("%Y-%m")}')
print(f'IPC          : {ipc.index.min().strftime("%Y-%m")} - {ipc.index.max().strftime("%Y-%m")}')
print(f'Base deflac. : {BASE.strftime("%Y-%m")} (IPC = {IPC_BASE:,.0f})')

In [ ]:
# ── Celda 3: Resultado Primario y Financiero ──────────────────────────
primario   = get_serie_total('XIV_RESULTADO_PRIMARIO')
financiero = get_serie_total('XV_RESULTADO_FINANCIERO')

prim_real = deflactar(primario)
fin_real  = deflactar(financiero)

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
colors = ['#27ae60' if v >= 0 else '#e74c3c' for v in prim_real]
ax.bar(primario.index, prim_real / 1e6, width=20, color=colors, alpha=0.85,
       label='Resultado Primario')
ax.plot(financiero.index, fin_real / 1e6, 'o-', color='#2c3e50',
        lw=1.5, ms=3, label='Resultado Financiero')
ax.axhline(0, color='black', lw=0.8)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.set_ylabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
fmt_meses(ax)
plt.tight_layout()
save_fig(fig, '01_resultado_primario_financiero')
plt.show()

# Tabla resumen anual
df_anual = pd.DataFrame({
    'Primario nominal (B)':   primario / 1e6,
    'Financiero nominal (B)': financiero / 1e6,
    'Primario real (B)':      prim_real / 1e6,
    'Financiero real (B)':    fin_real / 1e6,
})
df_anual = df_anual.groupby(df_anual.index.year).sum()
print(f'\nResumen anual - Sector Publico Total (billones ARS, real = precios {BASE.strftime("%b %Y")}):')
print(df_anual.round(1).to_string())

In [ ]:
# ── Celda 4: Ajuste fiscal - titulares y desglose ─────────────────────
print('Sector Publico Total (real, billones ARS):')
for concepto, label in [
    ('XIV_RESULTADO_PRIMARIO',                 'Resultado Primario'),
    ('XV_RESULTADO_FINANCIERO',                'Resultado Financiero'),
    ('XII_GASTOS_PRIMARIOS_DESPUES_FIGURAT',   'Gasto Primario'),
]:
    s = get_serie_total(concepto)
    s_real = pd.Series(deflactar(s), index=s.index)
    anual  = s_real.groupby(s_real.index.year).sum() / 1e6
    var24  = anual.get(2024, 0) - anual.get(2023, 0)
    var25  = anual.get(2025, 0) - anual.get(2023, 0)
    print(f'  {label}: 2022={anual.get(2022,0):.1f}  2023={anual.get(2023,0):.1f}  '
          f'2024={anual.get(2024,0):.1f}  2025={anual.get(2025,0):.1f} '
          f'| Var 23->24: {var24:+.1f}  Var 23->25: {var25:+.1f}')

print()
print('Desglose Administracion Nacional (real, billones ARS):')
conceptos_ajuste = {
    'I_INGRESOS_CORRIENTES':        'Ingresos corrientes',
    'II_GASTOS_CORRIENTES':         'Gastos corrientes',
    'II2_INTERESES':                '  Intereses',
    'II3_PRESTACIONES_SEG_SOCIAL':  '  Prestaciones Seg.Social',
    'II1a_REMUNERACIONES':          '  Remuneraciones',
    'II4b1_TRANSF_PROVINCIAS_CABA': '  Transf. Provincias/CABA',
    'II4b2_TRANSF_UNIVERSIDADES':   '  Universidades',
    'II4a_TRANSF_SECTOR_PRIVADO':   '  Transf. Privado (subsidios)',
    'V_GASTOS_CAPITAL':             'Gastos de capital',
    'XIV_RESULTADO_PRIMARIO':       'RESULTADO PRIMARIO',
}
rows = []
for cod, nombre in conceptos_ajuste.items():
    s = get_serie(cod)
    s_real = pd.Series(deflactar(s), index=s.index)
    for yr in [2022, 2023, 2024, 2025]:
        rows.append({'Concepto': nombre, 'Anio': yr,
                     'Real': s_real[s_real.index.year == yr].sum()})

df_ajuste = pd.DataFrame(rows).pivot(index='Concepto', columns='Anio', values='Real') / 1e6
df_ajuste['Var 23->24'] = df_ajuste[2024] - df_ajuste[2023]
df_ajuste['Var 23->25'] = df_ajuste[2025] - df_ajuste[2023]
df_ajuste['% ajuste 23->25'] = (
    df_ajuste['Var 23->25'] /
    abs(df_ajuste.loc['Gastos corrientes','Var 23->25'] +
        df_ajuste.loc['Gastos de capital','Var 23->25']) * 100
).round(1)
df_ajuste = df_ajuste.reindex(list(conceptos_ajuste.values()))
print(df_ajuste.round(1).to_string())

# Grafico: variacion 2023->2025 por componente
mask_det = df_ajuste.index.str.startswith('  ')
vals = df_ajuste.loc[mask_det, 'Var 23->25'].dropna().sort_values()
fig, ax = plt.subplots(figsize=(12, 5))
colors  = ['#27ae60' if v >= 0 else '#e74c3c' for v in vals]
bars = ax.barh(vals.index.str.strip(), vals.values, color=colors, alpha=0.85)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
for bar, val in zip(bars, vals.values):
    ax.text(val + (0.2 if val >= 0 else -0.2),
            bar.get_y() + bar.get_height()/2,
            f'{val:+.1f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=8)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
save_fig(fig, '02_ajuste_componentes_2023_2025')
plt.show()

In [ ]:
# ── Celda 5: Transferencias a provincias ──────────────────────────────
corr_tes = get_serie('II4b1_TRANSF_PROVINCIAS_CABA', 'tesoro_nacional')
cap_tes  = get_serie('V2a_TRANSF_CAPITAL_PROVINCIAS', 'tesoro_nacional')
corr_tot = get_serie('II4b1_TRANSF_PROVINCIAS_CABA', 'total_adm_nacional')
cap_tot  = get_serie('V2a_TRANSF_CAPITAL_PROVINCIAS', 'total_adm_nacional')

total_prov = corr_tot.add(cap_tot.reindex(corr_tot.index).fillna(0))
corr_real  = pd.Series(deflactar(corr_tes), index=corr_tes.index)
cap_real   = pd.Series(deflactar(cap_tes),  index=cap_tes.index)
total_prov_real = pd.Series(deflactar(total_prov), index=total_prov.index)

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
ax.stackplot(corr_real.index,
             corr_real.values / 1000,
             cap_real.reindex(corr_real.index).fillna(0).values / 1000,
             labels=['Corrientes (Tesoro)', 'Capital (Tesoro)'],
             colors=['#9b59b6', '#e67e22'], alpha=0.85)
ax.set_ylabel(f'Miles de MM ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
fmt_meses(ax)
plt.tight_layout()
save_fig(fig, '03_transferencias_provincias')
plt.show()

# Cuantificacion
gasto_tot  = get_serie_total('XII_GASTOS_PRIMARIOS_DESPUES_FIGURAT')
gasto_real = pd.Series(deflactar(gasto_tot), index=gasto_tot.index)
resumen    = pd.DataFrame({
    'Transf. prov. (B)': total_prov_real.groupby(total_prov_real.index.year).sum() / 1e6,
    'Gasto prim. total (B)': gasto_real.groupby(gasto_real.index.year).sum() / 1e6,
})
resumen['Prov./Gasto (%)'] = resumen['Transf. prov. (B)'] / resumen['Gasto prim. total (B)'] * 100

var_prov   = resumen.loc[2024,'Transf. prov. (B)'] - resumen.loc[2023,'Transf. prov. (B)']
var_prov25 = resumen.loc[2025,'Transf. prov. (B)'] - resumen.loc[2023,'Transf. prov. (B)']
var_gasto  = resumen.loc[2024,'Gasto prim. total (B)'] - resumen.loc[2023,'Gasto prim. total (B)']
var_gasto25= resumen.loc[2025,'Gasto prim. total (B)'] - resumen.loc[2023,'Gasto prim. total (B)']

print(resumen.round(1).to_string())
print(f'\n=== AJUSTE A PROVINCIAS ===')
print(f'2023->2024: {var_prov:+.1f} B ({var_prov/resumen.loc[2023,"Transf. prov. (B)"]*100:.0f}%) | '
      f'{var_prov/var_gasto*100:.1f}% del ajuste de gasto')
print(f'2023->2025: {var_prov25:+.1f} B ({var_prov25/resumen.loc[2023,"Transf. prov. (B)"]*100:.0f}%) | '
      f'{var_prov25/var_gasto25*100:.1f}% del ajuste de gasto')

In [ ]:
# ── Celda 6: Composicion del gasto corriente ──────────────────────────
comp_gasto = {
    'II3_PRESTACIONES_SEG_SOCIAL': 'Prestaciones Seg.Social',
    'II2_INTERESES':               'Intereses',
    'II1a_REMUNERACIONES':         'Remuneraciones',
    'II4a_TRANSF_SECTOR_PRIVADO':  'Subsidios',
    'II4b1_TRANSF_PROVINCIAS_CABA':'Transf. Provincias',
    'II4b2_TRANSF_UNIVERSIDADES':  'Universidades',
}
colors_comp = ['#e74c3c','#8e44ad','#3498db','#2ecc71','#9b59b6','#f39c12']

idx    = get_serie('II_GASTOS_CORRIENTES').index
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
bottom = np.zeros(len(idx))

for (cod, label), color in zip(comp_gasto.items(), colors_comp):
    vals = deflactar(get_serie(cod).reindex(idx).fillna(0)) / 1e6
    ax.bar(idx, vals, width=20, bottom=bottom, label=label, color=color, alpha=0.85)
    bottom += vals

ax.plot(idx, deflactar(get_serie('II_GASTOS_CORRIENTES')) / 1e6,
        'k-', lw=1.5, label='Total gastos corrientes')
ax.set_ylabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(axis='y', alpha=0.3)
fmt_meses(ax)
plt.tight_layout()
save_fig(fig, '04_composicion_gasto_corriente')
plt.show()

In [ ]:
# ── Celda 7: Composicion de ingresos ──────────────────────────────────
idx = get_serie('I_INGRESOS_CORRIENTES').index
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
bottom = np.zeros(len(idx))

for cod, label, color in [
    ('I1_INGRESOS_TRIBUTARIOS',    'Tributarios',    '#2ecc71'),
    ('I2_APORTES_SEG_SOCIAL',      'Seg. Social',    '#3498db'),
    ('I3_INGRESOS_NO_TRIBUTARIOS', 'No tributarios', '#f39c12'),
]:
    vals = deflactar(get_serie(cod).reindex(idx).fillna(0)) / 1e6
    ax.bar(idx, vals, width=20, bottom=bottom, label=label, color=color, alpha=0.85)
    bottom += vals

ax.plot(idx, deflactar(get_serie('I_INGRESOS_CORRIENTES')) / 1e6,
        'k-', lw=1.5, label='Total')
ax.set_ylabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
fmt_meses(ax)
plt.tight_layout()
save_fig(fig, '05_composicion_ingresos')
plt.show()

In [ ]:
# ── Celda 8: Resultado por subsector ──────────────────────────────────
subsectores = {
    'tesoro_nacional':      'Tesoro Nacional',
    'inst_seg_social':      'Seg. Social (ANSES)',
    'org_descentralizados': 'Org. Descentralizados',
    'rec_afectados':        'Rec. Afectados',
    'pami_fdos_otros':      'PAMI + Fondos Fiduciarios',
}
colors_ss = ['#e74c3c','#2ecc71','#3498db','#f39c12','#95a5a6']

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
for (ss, label), color in zip(subsectores.items(), colors_ss):
    s = get_serie('XIV_RESULTADO_PRIMARIO', ss)
    if len(s) == 0: continue
    ax.plot(s.index, deflactar(s) / 1e6, label=label, lw=1.5, color=color)

s_tot = get_serie_total('XIV_RESULTADO_PRIMARIO')
ax.plot(s_tot.index, deflactar(s_tot) / 1e6,
        'k--', lw=2, label='Total Sector Publico')
ax.axhline(0, color='black', lw=0.8, ls=':')
ax.set_ylabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
fmt_meses(ax)
plt.tight_layout()
save_fig(fig, '06_resultado_por_subsector')
plt.show()

In [ ]:
# ── Celda 9: Intereses de deuda ───────────────────────────────────────
intereses = get_serie_total('II2_INTERESES')
int_real  = deflactar(intereses)

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
ax.bar(intereses.index, int_real / 1e6, width=20, color='#8e44ad', alpha=0.85)
ax.set_ylabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.grid(axis='y', alpha=0.3)
fmt_meses(ax)
plt.tight_layout()
save_fig(fig, '07_intereses_deuda')
plt.show()

In [ ]:
# ── Celda 10: Exportar Excel + ZIP ────────────────────────────────────
EXCEL_FILE = 'analisis_fiscal_resultados.xlsx'
ZIP_FILE   = 'analisis_fiscal.zip'

# Serie mensual KPIs
idx_base = get_serie_total('XIV_RESULTADO_PRIMARIO').index
rows_m   = {'fecha': idx_base}
for nombre, cod in [
    ('ingresos_corrientes',  'I_INGRESOS_CORRIENTES'),
    ('gastos_corrientes',    'II_GASTOS_CORRIENTES'),
    ('intereses',            'II2_INTERESES'),
    ('prestaciones',         'II3_PRESTACIONES_SEG_SOCIAL'),
    ('resultado_primario',   'XIV_RESULTADO_PRIMARIO'),
    ('resultado_financiero', 'XV_RESULTADO_FINANCIERO'),
]:
    s = get_serie_total(cod)
    rows_m[f'{nombre}_nominal_MM_ARS'] = s.reindex(idx_base).values
    rows_m[f'{nombre}_real_MM_ARS']    = deflactar(s.reindex(idx_base))
df_serie = pd.DataFrame(rows_m)

# Resumen anual
df_anual_exp = df_serie.copy()
df_anual_exp['anio'] = pd.to_datetime(df_anual_exp['fecha']).dt.year
df_anual_exp = df_anual_exp.groupby('anio').sum(numeric_only=True)

# Transferencias a provincias
prov_rows = []
for cod, tipo in [('II4b1_TRANSF_PROVINCIAS_CABA','Corrientes'),
                  ('V2a_TRANSF_CAPITAL_PROVINCIAS','Capital')]:
    for ss in ['tesoro_nacional','total_adm_nacional']:
        s = get_serie(cod, ss)
        if len(s) == 0: continue
        prov_rows.append(pd.DataFrame({
            'fecha': s.index, 'tipo': tipo, 'subsector': ss,
            'nominal_MM_ARS': s.values, 'real_MM_ARS': deflactar(s),
        }))
df_prov_exp = pd.concat(prov_rows).sort_values(['fecha','tipo'])

# Exportar Excel
with pd.ExcelWriter(EXCEL_FILE, engine='openpyxl') as writer:
    df_serie.to_excel(writer,      sheet_name='Serie_mensual',       index=False)
    df_anual_exp.to_excel(writer,  sheet_name='Resumen_anual',       index=True)
    df_prov_exp.to_excel(writer,   sheet_name='Transferencias_prov', index=False)
    df_ajuste.reset_index().to_excel(writer, sheet_name='Ajuste_componentes', index=False)

print(f'Excel generado: {EXCEL_FILE}')

# Empaquetar todo en ZIP
with zipfile.ZipFile(ZIP_FILE, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(EXCEL_FILE)
    for g in GRAFICOS:
        if os.path.exists(g):
            zf.write(g)

contenido = [EXCEL_FILE] + GRAFICOS
print(f'\nZIP generado: {ZIP_FILE}')
print(f'Contenido ({len(contenido)} archivos):')
for f in contenido:
    size = os.path.getsize(f) / 1024
    print(f'  {f}  ({size:.0f} KB)')

# Descargar
if IN_COLAB:
    from google.colab import files
    files.download(ZIP_FILE)
    print('\nDescarga iniciada')
else:
    print(f'\nArchivo guardado en: {os.path.abspath(ZIP_FILE)}')